In [1]:
import pandas as pd
import numpy as np
import json
import os
import logging
import sys

# Set up logging for console output
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', stream=sys.stdout)

# --- CONFIGURATION ---
# Previous master file (contains data up to Round 10) [3]
MASTER_FILE_PATH_OLD = 'bbo_master_w11.csv'

# New master file for Week 12 (will contain data up to Round 11)
MASTER_FILE_PATH_NEW = 'bbo_master_w12.csv'

ADD_DATA_DIR = 'add_data'

# Files for the new data point (Round 11 data) [4]
INPUTS_FILE = 'week11_clean_inputs.json'
OUTPUTS_FILE = 'week11_clean_outputs.json'

NUM_FUNCTIONS = 8
CURRENT_ROUND = 11 # Appending Round 11 data

# --- Function Dimensionality Mapping ---
# Defined in project sources: F1-F2 (2D), F3 (3D), F4-F5 (4D), F6 (5D), F7 (6D), F8 (8D) [5, 6]
FUNCTION_DIMS = {
    1: 2, 2: 2, 3: 3, 4: 4, 
    5: 4, 6: 5, 7: 6, 8: 8 
}

def load_json_file(file_path):
    """Loads a JSON file or returns None with a warning."""
    full_path = os.path.join(ADD_DATA_DIR, file_path)
    if not os.path.exists(full_path):
        logging.error(f"ERROR: Required file '{full_path}' not found.")
        return None
    try:
        with open(full_path, 'r') as f:
            data = json.load(f)
            logging.info(f"Successfully loaded {file_path}.")
            return data
    except Exception as e:
        logging.error(f"An unexpected error occurred loading {file_path}: {e}")
        return None

def create_master_data():
    logging.info("*"*50)
    logging.info(f"--- Starting BBO Master File Creation for Round {CURRENT_ROUND} ---")

    # 1. Load the previous master file
    if not os.path.exists(MASTER_FILE_PATH_OLD):
        logging.error(f"Master file '{MASTER_FILE_PATH_OLD}' not found.")
        return None

    try:
        df_master_old = pd.read_csv(MASTER_FILE_PATH_OLD)
        logging.info(f"Loaded {len(df_master_old)} rows from {MASTER_FILE_PATH_OLD}.")
        
        # Ensure the 'Round' column is present and correctly reflects history (R0-R10) [7]
        num_rounds_after_r0 = (len(df_master_old) - 80) // 8
        round_list = [0] * 80 
        for r in range(1, num_rounds_after_r0 + 1):
            round_list.extend([r] * 8)
        
        if len(round_list) == len(df_master_old):
            df_master_old['Round'] = round_list
            logging.info(f"FIX APPLIED: 'Round' column verified (R0 to R{num_rounds_after_r0}).")
    except Exception as e:
        logging.error(f"Failed to process master CSV: {e}")
        return None

    # 2. Load the new inputs and outputs (Round 11)
    inputs_list = load_json_file(INPUTS_FILE)
    outputs_array = load_json_file(OUTPUTS_FILE)

    if inputs_list is None or outputs_array is None:
        logging.error("Aggregation aborted due to missing JSON files.")
        return None

    # 3. Construct the new 8 rows
    new_data = []
    cols_to_use = df_master_old.columns.tolist()

    for i in range(NUM_FUNCTIONS):
        f_id = i + 1
        dim = FUNCTION_DIMS[f_id]
        
        # Get inputs and outputs [8]
        x_values_raw = inputs_list[i] if i < len(inputs_list) else []
        score = outputs_array[i] if i < len(outputs_array) else np.nan

        # Validation and Padding logic [9, 10]
        if len(x_values_raw) > dim:
            x_values_to_use = x_values_raw[:dim]
        elif len(x_values_raw) < dim:
            x_values_to_use = x_values_raw + [0.5] * (dim - len(x_values_raw))
        else:
            x_values_to_use = x_values_raw

        # Pad the row with NaNs for X-columns beyond 'dim' (up to 8D)
        x_values = x_values_to_use + [np.nan] * (8 - dim)

        row_data = {
            'Function ID': f_id,
            'Y': score,
            'Round': CURRENT_ROUND
        }
        for d in range(8):
            row_data[f'X{d+1}'] = x_values[d]
        new_data.append(row_data)

    # 4. Concatenate and Save
    df_new_rows = pd.DataFrame(new_data, columns=cols_to_use)
    df_master_new = pd.concat([df_master_old, df_new_rows], ignore_index=True)
    df_master_new.to_csv(MASTER_FILE_PATH_NEW, index=False)

    # 5. Verification
    total_rows = len(df_master_new)
    # Expected: 80 initial + 11 rounds (1-11) * 8 functions = 168 rows [2, 11]
    logging.info("\n" + "*"*50)
    logging.info(f"SUCCESS: New master data file '{MASTER_FILE_PATH_NEW}' created.")
    logging.info(f"Total rows: {total_rows} (Expected 168).")
    logging.info(f"Verification: There are now {total_rows / 8:.1f} data points per function.")
    logging.info("*"*50)
    return df_master_new

if __name__ == '__main__':
    df_final = create_master_data()
    if df_final is not None:
        print("\n--- Tail of new data (Round 11) ---")
        print(df_final[df_final['Round'] == CURRENT_ROUND][['Function ID', 'Y']])

INFO: **************************************************
INFO: --- Starting BBO Master File Creation for Round 11 ---
INFO: Loaded 160 rows from bbo_master_w11.csv.
INFO: FIX APPLIED: 'Round' column verified (R0 to R10).
INFO: Successfully loaded week11_clean_inputs.json.
INFO: Successfully loaded week11_clean_outputs.json.
INFO: 
**************************************************
INFO: SUCCESS: New master data file 'bbo_master_w12.csv' created.
INFO: Total rows: 168 (Expected 168).
INFO: Verification: There are now 21.0 data points per function.
INFO: **************************************************

--- Tail of new data (Round 11) ---
     Function ID             Y
160            1  1.700777e-15
161            2  7.281983e-01
162            3 -1.013003e-02
163            4 -1.845638e+00
164            5  8.662405e+03
165            6 -4.511237e-01
166            7  4.174490e-01
167            8  3.562991e+00
